In [1]:
import json
import ast
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch
from sklearn.model_selection import train_test_split

In [ ]:
import kagglehub

path = kagglehub.dataset_download("neelshah18/arxivdataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/neelshah18/arxivdataset


In [3]:
def load_arxiv_json_to_dataframe(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    records = []
    for item in data:
        title = item.get('title', '').strip()
        abstract = item.get('summary', '').strip()
        tag_str = item.get('tag', '[]')
        tags_list = ast.literal_eval(tag_str)
        tags = [tag['term'] for tag in tags_list if isinstance(tag, dict) and 'term' in tag]
        records.append({'title': title, 'abstract': abstract, 'tags': tags})
    return pd.DataFrame(records)

data = load_arxiv_json_to_dataframe(path + '/arxivData.json')
X = data[['title', 'abstract']]
y = data[['tags']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

labels = sorted(set(tag for tags in data['tags'] for tag in tags))
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

def encode_labels(tags):
    vec = [0.0] * num_labels
    for tag in tags:
        if tag in label2id:
            vec[label2id[tag]] = 1.0
    return vec

y_train['labels'] = y_train['tags'].apply(encode_labels)
y_test['labels'] = y_test['tags'].apply(encode_labels)

X_train['text'] = X_train['title'] + ' ' + X_train['abstract']
X_test['text'] = X_test['title'] + ' ' + X_test['abstract']

train_dataset = Dataset.from_dict({'text': X_train['text'].tolist(), 'labels': y_train['labels'].tolist()})
test_dataset = Dataset.from_dict({'text': X_test['text'].tolist(), 'labels': y_test['labels'].tolist()})

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-cased", 
    use_fast=True,
    cache_dir="/kaggle/working/hf_cache"
)

def preprocess(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)

train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)
train_dataset.set_format('torch')
test_dataset.set_format('torch')

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/32800 [00:00<?, ? examples/s]

Map:   0%|          | 0/8200 [00:00<?, ? examples/s]

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-cased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True,
    cache_dir="/kaggle/working/hf_cache"
)

training_args = TrainingArguments(
    output_dir='results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='no',
    save_strategy='no',
    fp16=False,
    report_to="none",
    dataloader_num_workers=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()
trainer.save_model('distilbert_arxiv_tags')
tokenizer.save_pretrained('distilbert_arxiv_tags')

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **

Step,Training Loss
500,0.106995
1000,0.006235
1500,0.005136
2000,0.004570
2500,0.004314
3000,0.004114
3500,0.003991
4000,0.004013
4500,0.003703
5000,0.003594


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert_arxiv_tags/tokenizer_config.json',
 'distilbert_arxiv_tags/tokenizer.json')

In [6]:
def predict_top95(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.sigmoid(outputs.logits)[0].cpu().numpy()
    sorted_idx = np.argsort(probs)[::-1]
    cum = 0.0
    selected = []
    for idx in sorted_idx:
        cum += probs[idx]
        selected.append(id2label[idx])
        if cum > 0.95:
            break
    return selected

In [7]:
y_true = [set(tags) for tags in y_test['tags']]
y_pred = []

for text in X_test['text']:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.sigmoid(outputs.logits)[0].cpu().numpy()
    sorted_idx = np.argsort(probs)[::-1]
    cum = 0.0
    selected = []
    for idx in sorted_idx:
        cum += probs[idx]
        selected.append(id2label[idx])
        if cum > 0.95:
            break
    y_pred.append(set(selected))

f1s = []
for t, p in zip(y_true, y_pred):
    prec = len(t & p) / len(p) if len(p) > 0 else 0
    rec = len(t & p) / len(t) if len(t) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    f1s.append(f1)

print(np.mean(f1s))

0.6964032017614475
